# Galerie modèles pré-entraînés — [01 🟢] Sentiment d'avis clients (FluxPay)

> Étagère asynchrone **optionnelle**. Pas de livrable, pas de note.
> Geste travaillé : **utiliser un modèle pré-entraîné** (≠ en entraîner un).

**Contexte fictif.** *FluxPay*, une fintech, reçoit des centaines d'avis clients par semaine. On te demande de **trier automatiquement** le sentiment (positif / négatif) — **sans entraîner de modèle** : tu mobilises un modèle déjà entraîné sur HuggingFace.

C'est le **geste d'intégrateur** : réutiliser une capacité existante. On suit un **pattern en 4 temps** :
1. **Choisir** le modèle (model card, licence, taille)
2. **Charger** via `pipeline()`
3. **Tester + mesurer** (latence, taille)
4. **Comparer** à une alternative plus sobre → décider


## Setup

> ⚠️ **Premier lancement = téléchargement du modèle** (connexion requise une fois, puis mise en cache locale). Tout tourne sur **CPU**, pas besoin de GPU.

```bash
pip install "transformers>=4.40" torch sentence-transformers scikit-learn pandas
```

In [ ]:
import time
from pathlib import Path
from transformers import pipeline

## [1] Choisir le modèle

Avant de coder, on lit la **model card** (cf. `panorama_huggingface_hub.md`, checklist d'adoption) :

- **Modèle** : `nlptown/bert-base-multilingual-uncased-sentiment`
- **Tâche** : sentiment, note de **1 à 5 étoiles**
- **Langue** : multilingue (dont le **français**) ✅
- **Licence** : MIT ✅ (usage commercial OK)
- **Taille** : ~670 Mo (tient en mémoire CPU)


## [2] Charger via `pipeline()`

In [ ]:
MODELE = "nlptown/bert-base-multilingual-uncased-sentiment"
t0 = time.perf_counter()
clf = pipeline("sentiment-analysis", model=MODELE)
print(f"Modèle chargé en {time.perf_counter()-t0:.1f}s")

## [3] Tester + mesurer la latence

Un modèle pré-entraîné tourne **immédiatement**, sans entraînement.

In [ ]:
avis = [
    "Service client remarquable, je recommande vivement !",
    "Livraison catastrophique, colis arrivé cassé.",
    "Produit correct sans plus.",
]
t0 = time.perf_counter()
resultats = clf(avis)
latence = (time.perf_counter()-t0)*1000
for a, r in zip(avis, resultats):
    print(f"{r['label']:>8}  (score {r['score']:.2f})  | {a}")
print(f"\nLatence totale : {latence:.0f} ms pour {len(avis)} avis")

**Résultat attendu** : `5 stars` / `1 star` / `3 stars`. Le modèle capte le sentiment français sans qu'on lui ait montré le moindre exemple FluxPay — c'est tout l'intérêt du pré-entraîné.


## [4] Sobriété — le modèle est-il justifié ?

Fil rouge de la galerie : un modèle pré-entraîné est **souvent le choix sobre** (zéro entraînement), **mais** il a un coût d'inférence et une taille. On se donne deux repères :
- **la taille du modèle** (mémoire/disque)
- **la latence par avis**

et on se demande : *un classifieur classique maison (TF-IDF + régression logistique) ferait-il aussi bien, pour moins cher ?*

In [ ]:
# Taille approximative du modèle en cache + latence par avis
from huggingface_hub import scan_cache_dir
try:
    taille_mo = sum(r.size_on_disk for r in scan_cache_dir().repos
                    if MODELE.split('/')[-1] in r.repo_id) / 1e6
    print(f"Taille en cache : ~{taille_mo:.0f} Mo")
except Exception:
    print("Taille en cache : ~670 Mo (ordre de grandeur)")
print(f"Latence : ~{latence/len(avis):.0f} ms / avis")

## 🤔 Question réflexive — pré-entraîné vs classique maison

Le modèle pré-entraîné gagne **sans aucune donnée ni entraînement** : imbattable pour un POC ou un faible volume. Mais à **gros volume**, ~670 Mo et ~quelques dizaines de ms/avis peuvent peser.

Un **TF-IDF + régression logistique** entraîné sur quelques milliers d'avis FluxPay labellisés serait **~1000× plus léger**, quasi instantané et **explicable** — mais il faut les labels. 

> 💡 **La bonne question n'est jamais « quel est le plus puissant ? » mais « quel est le plus simple qui résout le besoin ? »** Pré-entraîné si tu n'as pas de labels / faible volume ; classique maison si tu as des données et un fort volume.


## 📝 Verdict — à formuler en 3 lignes

Pour FluxPay, recommanderais-tu le modèle pré-entraîné ou un classifieur maison ? Sur quels critères (volume d'avis, disponibilité de labels, contrainte de latence) ? Écris ton verdict — c'est exactement le raisonnement attendu en M4-B2 et M8.